## Prepare a jsonl dataset file

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import mlcroissant as mlc

hint_path = Path("data/nanogpt_speedrun_knowledge_in_levels")
workspace_path = Path("workspace_templates/nanogpt_speedrun")

target_hint_path = Path("hints")
target_workspace_path = Path("workspaces")

records = []
for i in range(1, 21):
    record_info = {
        "id": i,
        "level_0_hint": (hint_path / f"record_{i}" / "level_0_diff.txt").read_text(),
        "level_1_hint": (hint_path / f"record_{i}" / "level_1_pseudo.txt").read_text(),
        "level_2_hint": (hint_path / f"record_{i}" / "level_2_description.txt").read_text(),
        "level_3_hint": (hint_path / f"record_{i}" / "level_3_tweets.txt").read_text(),
        "level_4_hint": (hint_path / f"record_{i}" / "level_4_human_tweets.txt").read_text(),
        "level_5_hint": (hint_path / f"record_{i}" / "level_5_paper.txt").read_text(),
        "current_results": (workspace_path / f"record_{i}" / "results.json").read_text(),
        "current_train_script": (workspace_path / f"record_{i}" / "train_gpt2.py").read_text(),
        "level_0_hint_path": str(target_hint_path / f"record_{i}" / "level_0_diff.txt"),
        "level_1_hint_path": str(target_hint_path / f"record_{i}" / "level_1_pseudo.txt"),
        "level_2_hint_path": str(target_hint_path / f"record_{i}" / "level_2_description.txt"),
        "level_3_hint_path": str(target_hint_path / f"record_{i}" / "level_3_tweets.txt"),
        "level_4_hint_path": str(target_hint_path / f"record_{i}" / "level_4_human_tweets.txt"),
        "level_5_hint_path": str(target_hint_path / f"record_{i}" / "level_5_paper.txt"),
        "current_results_path": str(target_workspace_path / f"record_{i}" / "results.json"),
        "current_train_script_path": str(target_workspace_path / f"record_{i}" / "train_gpt2.py"),
        "target_time": json.loads(
            (workspace_path / f"record_{i+1}" / "results.json").read_text()
        )["metrics"]["train_time"],
    }
    if i in [2, 3]:
        record_info["level_9_hint"] = (hint_path / f"record_{i}" / "level_9_muon.txt").read_text()
        record_info["level_9_hint_path"] = str(target_hint_path / f"record_{i}" / "level_9_muon.txt")
    if i in [11, 12]:
        record_info["level_9_hint"] = (hint_path / f"record_{i}" / "level_9_flexattn.txt").read_text()
        record_info["level_9_hint_path"] = str(target_hint_path / f"record_{i}" / "level_9_flexattn.txt")
    records.append(record_info)

records_df = pd.DataFrame(records)
records_df.to_json("data.jsonl", orient="records", lines=True)

## Generate a Croissant metadata file

In [ ]:
# FileObjects and FileSets define the resources of the dataset.
distribution = [
    mlc.FileObject(
        id="nanogpt_records.zip",
        name="nanogpt_records.zip",
        description="Zip file containing the nanoGPT speedrun records",
         # TODO: change hosted zip location
        content_url=(
            "<TODO>nanogpt_records.zip"
        ),
        encoding_formats=["application/zip"],
        sha256="17d1ba6770a68ab3a0458517f97b79960d1a8d8a5775a7aafaf9f70e591121f3",
    ),
    mlc.FileObject(
        id="data",
        name="data",
        description="",
        contained_in=["nanogpt_records.zip"],
        content_url="data.jsonl",
        encoding_formats=["application/jsonlines"],
    ),
]
record_sets = [
    # RecordSets contains records in the dataset.
    mlc.RecordSet(
        id="nanogpt_records",
        name="nanogpt_records",
        # Each record has one or many fields...
        fields=[
            # Fields can be extracted from the FileObjects/FileSets.
            mlc.Field(
                id="nanogpt_records/id",
                name="id",
                description="The nanoGPT speedrun record ID",
                data_types=mlc.DataType.INTEGER,
                source=mlc.Source(
                    file_object="data",
                    extract=mlc.Extract(column="id"),
                ),
            ),
            *[
                mlc.Field(
                    id=f"nanogpt_records/{field_name}",
                    name=field_name,
                    description=description,
                    data_types=mlc.DataType.TEXT,
                    source=mlc.Source(
                        file_object="data",
                        # Extract the field from the column of a FileObject/FileSet:
                        extract=mlc.Extract(column=field_name),
                    ),
                ) for field_name, description in {
                    "level_0_hint": "Level 0 hint: code diff between current record and next record", 
                    "level_1_hint": "Level 1 hint: pseduocode for the next record", 
                    "level_2_hint": "Level 2 hint: detailed text description for the next record", 
                    "level_3_hint": "Level 3 hint: LLM generated tweet for the next record", 
                    "level_4_hint": "Level 4 hint: human generated tweet for the next record", 
                    "level_5_hint": "Level 5 hint: the next record described in a paper format",
                    "level_9_hint": "Level 9 hint: external knowledge describing technique used by the next record, if applicable",
                    "current_results": "JSON output for the current record",
                    "current_train_script": "Python script used to train the model for the current record",
                    "level_0_hint_path": "The local path to the file containing the level 0 hint", 
                    "level_1_hint_path": "The local path to the file containing the level 1 hint", 
                    "level_2_hint_path": "The local path to the file containing the level 2 hint", 
                    "level_3_hint_path": "The local path to the file containing the level 3 hint", 
                    "level_4_hint_path": "The local path to the file containing the level 4 hint", 
                    "level_5_hint_path": "The local path to the file containing the level 5 hint", 
                    "level_9_hint_path": "The local path to the file containing the level 9 hint, if applicable", 
                    "current_results_path": "The local path to the JSON output for the current record",
                    "current_train_script_path": "The local path to the Python script used to train the model for the current record",
                }.items()
            ],
            mlc.Field(
                id="nanogpt_records/target_time",
                name="target_time",
                description="The target runtime that the next record achieved",
                data_types=mlc.DataType.INTEGER,
                source=mlc.Source(
                    file_object="data",
                    extract=mlc.Extract(column="target_time"),
                ),
            ),
        ],
    )
]

# Metadata contains information about the dataset.
metadata = mlc.Metadata(
    name="nanogpt_speedrun",
    # Descriptions can contain plain text or markdown.
    description=(
        "This dataset contains the nanoGPT reproducibility benchmark, a series of records "
        "from the nanoGPT speedrun accompanied by hints describing how to reach the next record. "
        "The goal of this benchmark is to evaluate the ability of AI agents to ingest knowledge and "
        "implement improvements to LLM training algorithms."
    ),
    cite_as=(
        "@article{<TO_FILL_OUT>, title={<TO_FILL_OUT>}, author={<TO_FILL_OUT> <TO_FILL_OUT>}, year={2025},"
        " eprint={<TO_FILL_OUT>}, archivePrefix={arXiv}, primaryClass={cs.CL} }"
    ),
    # url="",  # TODO: change to something else
    license=["<TO_FILL_OUT>"],
    version=1,
    # Bug: mlcroissant converts this to a datetime object (even when a string is specified) and then json.dumps fails
    # date_published=datetime(2025, 5, 15),
    distribution=distribution,
    record_sets=record_sets,
)

In [ ]:
print(metadata.issues.report())

In [ ]:
import json

with open("croissant.json", "w") as f:
    content = metadata.to_json()
    content = json.dumps(content, indent=2)
    print(content)
    f.write(content)
    f.write("\n")  # Terminate file with newline

## Load rows from Croissant metadata file

In [ ]:
dataset = mlc.Dataset(jsonld="croissant.json")

In [ ]:
records = dataset.records(record_set="nanogpt_records")

for i, record in enumerate(records):
    print(record)